In [1]:
# Load the statistical analysis libraries and feature-engineered dataset.

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_parquet('../data/processed/vehicles_features.parquet')

In [2]:
# Test whether gasoline vehicle MPG differs significantly across drivetrains.

ice = (
    df[df['powertrain'] == 'Gasoline']
    .dropna(subset=['comb08', 'drive'])
)

groups = [
    group['comb08'].values
    for name, group in ice.groupby('drive')
]

f_stat, p_value = stats.f_oneway(*groups)

print(f"ANOVA: F = {f_stat:.2f}, p = {p_value:.4g}")

print(
    ice.groupby('drive')['comb08']
    .mean()
    .sort_values(ascending=False)
)

ANOVA: F = 4309.47, p = 0
drive
Front-Wheel Drive             24.024476
All-Wheel Drive               21.485399
4-Wheel Drive                 19.168367
Part-time 4-Wheel Drive       18.126819
Rear-Wheel Drive              17.881075
4-Wheel or All-Wheel Drive    16.739409
2-Wheel Drive                 16.171021
Name: comb08, dtype: float64


The tests below are exploratory associations. Repeated configurations, model-year effects, unequal variances, and multiple comparisons limit inference. A pre/post-2012 difference does not identify a policy effect. Printed p = 0 means numerical underflow, not an exactly zero probability.


In [3]:
h_stat, p_value_kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H = {h_stat:.2f}, p = {p_value_kw:.4g}")

Kruskal-Wallis: H = 17161.26, p = 0


In [4]:
trans_data = df[df['powertrain'] == 'Gasoline'].dropna(subset=['comb08', 'transmission_type'])
groups_trans = [g['comb08'].values for _, g in trans_data.groupby('transmission_type')]

f_stat, p_value = stats.f_oneway(*groups_trans)
print(f"ANOVA: F = {f_stat:.2f}, p = {p_value:.4g}")
print(trans_data.groupby('transmission_type')['comb08'].mean().sort_values(ascending=False))

ANOVA: F = 1414.45, p = 0
transmission_type
CVT          27.879334
Manual       21.193742
Automatic    19.687423
Name: comb08, dtype: float64


In [5]:
cars = df[(df['powertrain'] == 'Gasoline') & (df['segment'] == 'Car')].dropna(subset=['comb08'])

before = cars[cars['year'] < 2012]['comb08']
after = cars[cars['year'] >= 2012]['comb08']

t_stat, p_value = stats.ttest_ind(after, before, equal_var=False)

print(f"Before 2012 mean: {before.mean():.2f} (n={len(before)})")
print(f"After 2012 mean:  {after.mean():.2f} (n={len(after)})")
print(f"t = {t_stat:.2f}, p = {p_value:.4g}")

Before 2012 mean: 21.19 (n=13524)
After 2012 mean:  23.87 (n=6889)
t = 37.23, p = 2.951e-287


In [6]:
pooled_std = np.sqrt(((len(before) - 1) * before.var() + (len(after) - 1) * after.var()) / (len(before) + len(after) - 2))
cohens_d = (after.mean() - before.mean()) / pooled_std
print(f"Cohen's d = {cohens_d:.2f}")

Cohen's d = 0.58


In [7]:
corr_data = df[df['powertrain'] == 'Gasoline'][['displ', 'cylinders', 'comb08']].dropna()

print(corr_data.corr())

r, p = stats.pearsonr(corr_data['displ'], corr_data['comb08'])
print(f"displacement vs combined MPG: r = {r:.3f}, p = {p:.4g}")

              displ  cylinders    comb08
displ      1.000000   0.904393 -0.777713
cylinders  0.904393   1.000000 -0.720298
comb08    -0.777713  -0.720298  1.000000
displacement vs combined MPG: r = -0.778, p = 0


In [8]:
contingency = pd.crosstab(df['segment'], df['powertrain'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-square = {chi2:.2f}, degrees of freedom = {dof}, p = {p_value:.4g}")
print(f"Expected cells below 5: {(expected < 5).sum()} / {expected.size}")


Chi-square = 3743.08, degrees of freedom = 48, p = 0
Expected cells below 5: 15 / 63
